<a href="https://colab.research.google.com/github/YAN-JINGHAO/TorchCode/blob/main/templates/09_causal_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.9 MB/s eta 0:00:00


In [2]:
import torch
import math

In [22]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    scores = torch.bmm(Q, K.transpose(-2, -1)) / math.sqrt(K.size(-1))
    mask = torch.triu(torch.ones_like(scores), diagonal=1).bool()
    mask_scores = scores.masked_fill(mask, float('-inf'))
    weights = torch.softmax(mask_scores, dim=-1)
    return torch.bmm(weights, V)

In [19]:
# ✅ SOLUTION

def causal_attention(Q, K, V):
    d_k = K.size(-1)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    S = scores.size(-1)
    mask = torch.triu(torch.ones(S, S, device=scores.device, dtype=torch.bool), diagonal=1)
    scores = scores.masked_fill(mask.unsqueeze(0), float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return torch.bmm(weights, V)

In [17]:
scores = torch.randn(1, 4, 4)
print(scores)
mask = torch.triu(torch.ones_like(scores), diagonal=1).bool()
print(mask)
mask_scores = scores.masked_fill(mask, float('-inf'))
print(mask_scores)
print(torch.softmax(mask_scores, dim=-1))

tensor([[[ 1.6459, -1.3602,  0.3446,  0.5199],
         [-2.6133, -1.6965, -0.2282,  0.2800],
         [ 0.2469,  0.0769,  0.3380,  0.4544],
         [ 0.4569, -0.8654,  0.7813, -0.9268]]])
tensor([[[False,  True,  True,  True],
         [False, False,  True,  True],
         [False, False, False,  True],
         [False, False, False, False]]])
tensor([[[ 1.6459,    -inf,    -inf,    -inf],
         [-2.6133, -1.6965,    -inf,    -inf],
         [ 0.2469,  0.0769,  0.3380,    -inf],
         [ 0.4569, -0.8654,  0.7813, -0.9268]]])
tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.2856, 0.7144, 0.0000, 0.0000],
         [0.3403, 0.2870, 0.3727, 0.0000],
         [0.3448, 0.0919, 0.4769, 0.0864]]])


In [23]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [24]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.0ms)
  ✅ [2/4] Future tokens don't affect past (2.7ms)
  ✅ [3/4] First position only sees itself (2.4ms)
  ✅ [4/4] Gradient flow (1.0ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (7.1ms total)
  Progress saved. Run status() to see your dashboard.



In [ ]:
# device=scores.device, dtype=torch.bool
# 先生成二维矩阵，再unsqueeze(0)补齐batch维度